In [ ]:
!pip install google-api-python-client pandas openpyxl

# download

In [ ]:
!pip install -U yt-dlp

### 1. Environment Setup & Dependency Installation
**Description:** This section initializes the working environment by downloading and updating all required libraries and tools for the project:
* `yt-dlp`: The core utility used to download audio and extract data from YouTube.
* `pandas` & `openpyxl`: Used for data structure manipulation and exporting extracted metadata into Excel workbooks.
* It also handles the creation of a dedicated `audios` folder to store downloaded media assets.

In [ ]:
!mkdir -p audios
!yt-dlp -v \
  -f bestaudio \
  --extract-audio \
  --audio-format wav \
  --write-auto-subs \
  --sub-lang ar \
  --convert-subs srt \
  --no-playlist \
  --write-sub \
  --output "audios/%(title).200s.%(ext)s" \
  "https://www.youtube.com/channel/" # add the link of the channel you want to download the audio from


### 2. Full Channel Extraction Pipeline
**Description:** A comprehensive Python script designed to automatically process an entire YouTube channel from scratch:
1. Prompts the user to input a targeted YouTube channel URL.
2. Fetches a complete list of all videos hosted on that channel using `yt-dlp`.
3. Loops through each video to download its audio component (converted to `.wav`) and grab its Arabic subtitles (`.srt`).
4. Renames all downloaded assets sequentially (e.g., `audio001.wav`, `srt001.srt`).
5. Compiles core metadata (Title, URL, Upload Date, Duration) into a structured dataframe and exports it to `video_metadata.xlsx`.

In [ ]:
import subprocess
import json
import isodate
import pandas as pd
import os

# ================= CONFIGURATION =================
channel_url = input("Enter YouTube channel URL: ").strip()

output_folder = "audios"
os.makedirs(output_folder, exist_ok=True)

# ================= FETCH PLAYLIST METADATA =================
print("Fetching video metadata from channel...\n")

command = [
    "yt-dlp",
    "--flat-playlist",
    "--dump-json",
    channel_url
]

result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

videos = []

# ================= FETCH DETAILED METADATA FOR EACH VIDEO =================
for idx, line in enumerate(result.stdout.splitlines(), 1):
    try:
        video_info = json.loads(line)
        video_id = video_info.get("id")
        video_url = f"https://www.youtube.com/watch?v={video_id}"

        # Get detailed info about the video (including upload_date, duration)
        detailed_cmd = [
            "yt-dlp",
            "--skip-download",
            "--write-auto-sub",
            "--sub-lang", "ar",
            "--convert-subs", "srt",
            "--dump-json",
            video_url
        ]

        detailed_result = subprocess.run(detailed_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        detailed_info = json.loads(detailed_result.stdout)

        # Extract needed metadata
        title = detailed_info.get("title", "Unknown Title")

        upload_date = detailed_info.get("upload_date", "")
        if upload_date:
            upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:8]}"

        duration_seconds = detailed_info.get("duration", 0)
        duration = str(isodate.parse_duration(f"PT{duration_seconds}S")) if duration_seconds else ""

        # Prepare output file names
        audio_filename = f"audio{idx:03d}.wav"
        srt_filename = f"srt{idx:03d}.srt"

        # Download audio and subtitles with custom naming
        download_cmd = [
            "yt-dlp",
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "wav",
            "--write-auto-sub",
            "--sub-lang", "ar",
            "--convert-subs", "srt",
            "-o", os.path.join(output_folder, f"audio{idx:03d}.%(ext)s"),
            video_url
        ]

        print(f"Downloading [{idx}/{len(result.stdout.splitlines())}]: {title}")
        subprocess.run(download_cmd)

        # Rename downloaded subtitle file to srtXXX.srt if exists
        # yt-dlp usually names subtitle files as <title>.ar.srt
        for file in os.listdir(output_folder):
            if file.endswith(".srt") and file.startswith(title[:50]):
                os.rename(
                    os.path.join(output_folder, file),
                    os.path.join(output_folder, srt_filename)
                )
                break

        # Append info to videos list
        videos.append({
            "AudioFile": audio_filename,
            "SubtitleFile": srt_filename,
            "Title": title,
            "UploadDate": upload_date,
            "Duration": duration,
            "URL": video_url
        })

    except Exception as e:
        print(f"Error processing video {idx}: {e}")
        continue

# ================= SAVE METADATA TO EXCEL =================
df = pd.DataFrame(videos)
excel_filename = "video_metadata.xlsx"
df.to_excel(excel_filename, index=False)
print(f"\nMetadata saved to {excel_filename}")


### 3. Video Processing via External Text File Links
**Description:** Instead of scraping a full channel, this script reads specific YouTube URLs provided line-by-line from a local text file (`.txt`). It introduces the `youtube_transcript_api` library to natively query subtitles—prioritizing manual Arabic subtitles before automatically falling back to auto-generated transcripts.

In [ ]:

import subprocess
import json
import isodate
import pandas as pd
import os
import re
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter

# ================= CONFIGURATION =================
txt_file = input("Enter path to .txt file with YouTube URLs: ").strip()
output_folder = "audios"
os.makedirs(output_folder, exist_ok=True)

# ================= HELPER: EXTRACT VIDEO ID =================
def get_video_id(url):
    match = re.search(r'(?:v=|youtu\.be/|embed/|watch\?v=|\?v=)([^&]+)', url)
    return match.group(1) if match else None

# ================= LOAD VIDEO URLS =================
with open(txt_file, "r", encoding="utf-8") as f:
    video_urls = [line.strip() for line in f if line.strip()]

print(f"\nFound {len(video_urls)} video URLs.\n")

videos = []

# ================= PROCESS EACH VIDEO =================
for idx, video_url in enumerate(video_urls, 1):
    try:
        video_id = get_video_id(video_url)
        if not video_id:
            print(f"Invalid YouTube URL at line {idx}. Skipping.")
            continue

        # ========== METADATA WITH yt-dlp ==========
        metadata_cmd = [
            "yt-dlp",
            "--skip-download",
            "--dump-json",
            video_url
        ]
        metadata_result = subprocess.run(metadata_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        metadata = json.loads(metadata_result.stdout)

        title = metadata.get("title", f"Unknown Title {idx}")
        upload_date = metadata.get("upload_date", "")
        if upload_date:
            upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:8]}"
        duration_seconds = metadata.get("duration", 0)
        duration = str(isodate.parse_duration(f"PT{duration_seconds}S")) if duration_seconds else ""

        audio_filename = f"audio{idx:03d}.wav"
        srt_filename = f"srt{idx:03d}.srt"

        # ========== DOWNLOAD AUDIO ==========
        download_cmd = [
            "yt-dlp",
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "wav",
            "-o", os.path.join(output_folder, f"audio{idx:03d}.%(ext)s"),
            video_url
        ]
        print(f"Downloading audio [{idx}/{len(video_urls)}]: {title}")
        subprocess.run(download_cmd)

        # ========== FETCH SUBTITLES USING youtube-transcript-api ==========
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            transcript = None

            # Prefer manual Arabic subtitles
            for t in transcript_list:
                if t.language_code == 'ar':
                    transcript = t
                    break
            # Fallback to auto-generated Arabic
            if not transcript:
                for t in transcript_list:
                    if t.language_code.startswith('a.') and 'ar' in t.language_code:
                        transcript = t
                        break

            if transcript:
                transcript_data = transcript.fetch()
                formatter = SRTFormatter()
                srt_content = formatter.format_transcript(transcript_data)

                with open(os.path.join(output_folder, srt_filename), "w", encoding="utf-8") as f:
                    f.write(srt_content)
                print(f"→ Subtitles downloaded for {title}")
            else:
                print(f"→ No Arabic subtitles found for {title}")
                srt_filename = ""

        except Exception as sub_err:
            print(f"→ Subtitle error for {title}: {sub_err}")
            srt_filename = ""

        # ========== SAVE METADATA ==========
        videos.append({
            "AudioFile": audio_filename,
            "SubtitleFile": srt_filename,
            "Title": title,
            "UploadDate": upload_date,
            "Duration": duration,
            "URL": video_url
        })

    except Exception as e:
        print(f"Error processing video {idx}: {e}")
        continue

# ================= SAVE TO EXCEL =================
df = pd.DataFrame(videos)
excel_filename = "video_metadata.xlsx"
df.to_excel(excel_filename, index=False)
print(f"\nMetadata saved to {excel_filename}")


### 4. Subtitle API Verification (Single Video Test)
**Description:** A focused testing snippet used to verify that the transcript API is functioning correctly. It extracts the raw Video ID from a single hardcoded YouTube link using Regular Expressions (Regex) and attempts to fetch and save the Arabic transcript as an independent `.srt` file.

In [ ]:


!pip install youtube-transcript-api
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter
import re

video_url = 'https://www.youtube.com/'   # Replace with the actual URL

def get_video_id(url):
    """Extracts the YouTube video ID from a URL."""
    # Regex to extract video ID from various YouTube URL formats
    match = re.search(r'(?:v=|youtu\.be/|embed/|watch\?v=|\?v=)([^&]+)', url)
    return match.group(1) if match else None

video_id = get_video_id(video_url)

if video_id:
    try:
        # List available transcripts
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        
        # You can iterate through available transcripts to find the desired language
        # For simplicity, let's try to get English (en) or auto-generated English (a.en)
        
        transcript = None
        for t in transcript_list:
            if t.language_code == 'ar':
                transcript = t
                break
        if not transcript: # Try auto-generated if manual not found
            for t in transcript_list:
                if t.language_code.startswith('a.') and 'ar' in t.language_code:
                    transcript = t
                    break

        if transcript:
            actual_transcript_data = transcript.fetch()
            
            formatter = SRTFormatter()
            srt_formatted = formatter.format_transcript(actual_transcript_data)

            # Save to an SRT file
            with open(f"{video_id}_{transcript.language_code}.srt", "w", encoding="utf-8") as f:
                f.write(srt_formatted)
            print(f"SRT for video ID '{video_id}' ({transcript.language_code}) downloaded successfully!")
        else:
            print(f"No English (or auto-generated English) captions found for video ID '{video_id}'.")

    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("Could not extract video ID from the provided URL.")

### 5. Batch Subtitle Resolution from Excel Using Multi-Threading
**Description:** An advanced script built to read an existing Excel metadata sheet and download missing subtitles for the listed links. To prevent the program from freezing or hanging indefinitely due to network or API bottlenecks, it implements a custom multi-threaded wrapper (`TranscriptDownloadThread`) featuring:
* A strict execution timeout limit (`MAX_DOWNLOAD_TIME = 10` seconds).
* An automatic retry mechanism (`MAX_RETRIES = 3`).
* Error logging directly inside the Excel spreadsheet if subtitles are disabled or unavailable.

In [ ]:


import os
import pandas as pd
import re
import time
import threading
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

# ====== CONFIG ======
excel_path = input("Enter Excel file path: ").strip()
srt_folder = "srt"
os.makedirs(srt_folder, exist_ok=True)
MAX_DOWNLOAD_TIME = 10  # seconds
MAX_RETRIES = 3
RETRY_DELAY = 1  # seconds

# ====== GET VIDEO ID FROM URL ======
def get_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([^&?/]+)", url)
    return match.group(1) if match else None

# ====== SAFE DOWNLOAD THREAD WRAPPER ======
class TranscriptDownloadThread(threading.Thread):
    def __init__(self, video_id, language_code='ar'):
        super().__init__()
        self.video_id = video_id
        self.language_code = language_code
        self.result = None
        self.error = None

    def run(self):
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(self.video_id)
            for t in transcript_list:
                if t.language_code == self.language_code:
                    self.result = t.fetch()
                    return
            for t in transcript_list:
                if t.language_code.startswith('a.') and 'ar' in t.language_code:
                    self.result = t.fetch()
                    return
            self.error = NoTranscriptFound(self.video_id, language_codes=[self.language_code])
        except Exception as e:
            self.error = e

# ====== DOWNLOAD WITH RETRIES ======
def download_with_retries(video_id, language_code='ar'):
    for attempt in range(1, MAX_RETRIES + 1):
        thread = TranscriptDownloadThread(video_id, language_code)
        thread.start()
        thread.join(timeout=MAX_DOWNLOAD_TIME)
        if thread.is_alive():
            thread.error = TimeoutError(f"Timeout after {MAX_DOWNLOAD_TIME}s")
        if thread.error:
            print(f"  ⚠️ Retry {attempt}/{MAX_RETRIES} failed: {thread.error}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            else:
                raise thread.error
        else:
            return thread.result

# ====== LOAD EXCEL ======
df = pd.read_excel(excel_path)

if "Error" not in df.columns:
    df["Error"] = ""
if "SubtitleFile" not in df.columns:
    df["SubtitleFile"] = ""

# ====== PROCESS EACH ROW ======
for idx, row in df.iterrows():
    try:
        url = row['URL']
        video_id = get_video_id(url)
        if not video_id:
            df.at[idx, 'Error'] = "Invalid URL"
            continue

        new_srt_name = f"srt{idx+1:03d}.srt"
        df.at[idx, 'SubtitleFile'] = new_srt_name

        transcript_data = download_with_retries(video_id)
        formatter = SRTFormatter()
        srt_content = formatter.format_transcript(transcript_data)

        with open(os.path.join(srt_folder, new_srt_name), "w", encoding="utf-8") as f:
            f.write(srt_content)

        print(f"[{idx+1}] ✓ Downloaded: {new_srt_name}")
        df.at[idx, 'Error'] = ""

    except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable, TimeoutError) as e:
        error_msg = str(e)
        print(f"[{idx+1}] ✗ {error_msg}")
        df.at[idx, 'Error'] = error_msg

    except Exception as e:
        error_msg = str(e)
        print(f"[{idx+1}] ERROR: {error_msg}")
        df.at[idx, 'Error'] = error_msg

# ====== SAVE RESULT ======
df.to_excel("updated_video_metadata.xlsx", index=False)
print("\n✅ Finished. Output saved to updated_video_metadata.xlsx")


  ⚠️ Retry 1/3 failed: no element found: line 1, column 0
[1] ✓ Downloaded: srt001.srt
[2] ✓ Downloaded: srt002.srt
[3] ✓ Downloaded: srt003.srt
[4] ✓ Downloaded: srt004.srt
[5] ✓ Downloaded: srt005.srt
  ⚠️ Retry 1/3 failed: no element found: line 1, column 0
[6] ✓ Downloaded: srt006.srt
[7] ✓ Downloaded: srt007.srt
[8] ✓ Downloaded: srt008.srt
[9] ✓ Downloaded: srt009.srt
[10] ✓ Downloaded: srt010.srt
[11] ✓ Downloaded: srt011.srt
[12] ✓ Downloaded: srt012.srt
[13] ✓ Downloaded: srt013.srt
[14] ✓ Downloaded: srt014.srt
[15] ✓ Downloaded: srt015.srt
  ⚠️ Retry 1/3 failed: no element found: line 1, column 0
  ⚠️ Retry 2/3 failed: no element found: line 1, column 0
[16] ✓ Downloaded: srt016.srt
  ⚠️ Retry 1/3 failed: no element found: line 1, column 0
[17] ✓ Downloaded: srt017.srt
  ⚠️ Retry 1/3 failed: no element found: line 1, column 0
[18] ✓ Downloaded: srt018.srt
[19] ✓ Downloaded: srt019.srt
[20] ✓ Downloaded: srt020.srt
[21] ✓ Downloaded: srt021.srt
[22] ✓ Downloaded: srt022.srt
[

### 6. Enhanced and Error-Resilient Channel Processing
**Description:** A major architectural upgrade to the original channel script. It isolates audio downloads using `yt-dlp` and pairs it with native `youtube_transcript_api` queries. Wrapped entirely inside a robust `try-except` block, this version ensures that if a single video has blocked transcripts or is broken, the script records the error and safely proceeds to the next video without crashing the entire program execution.

In [ ]:
import subprocess
import json
import isodate
import pandas as pd
import os
import re
import time
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

# ================= CONFIGURATION =================
channel_url = input("Enter YouTube channel URL: ").strip()

audio_folder = "audios"
srt_folder = "srt"
os.makedirs(audio_folder, exist_ok=True)
os.makedirs(srt_folder, exist_ok=True)

# ================= UTILITY FUNCTIONS =================
def get_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([^&?/]+)", url)
    return match.group(1) if match else None

def download_transcript(video_id, language_code='ar', retries=10, delay=2):
    for attempt in range(retries):
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcript_list:
                if t.language_code == language_code:
                    return t.fetch()
            for t in transcript_list:
                if t.language_code.startswith('a.') and 'ar' in t.language_code:
                    return t.fetch()
            raise NoTranscriptFound(video_id, language_codes=[language_code])
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                raise e

# ================= FETCH PLAYLIST METADATA =================
print("Fetching video metadata from channel...\n")

command = [
    "yt-dlp",
    "--flat-playlist",
    "--dump-json",
    channel_url
]

result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
playlist_items = result.stdout.strip().splitlines()

videos = []

# ================= PROCESS EACH VIDEO =================
for idx, line in enumerate(playlist_items, 1):
    try:
        video_info = json.loads(line)
        video_id = video_info.get("id")
        video_url = f"https://www.youtube.com/watch?v={video_id}"

        # Get detailed metadata
        detailed_cmd = [
            "yt-dlp",
            "--skip-download",
            "--dump-json",
            video_url
        ]
        detailed_result = subprocess.run(detailed_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        detailed_info = json.loads(detailed_result.stdout)

        title = detailed_info.get("title", f"Video {idx}")
        upload_date = detailed_info.get("upload_date", "")
        if upload_date:
            upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:8]}"
        duration_seconds = detailed_info.get("duration", 0)
        duration = str(isodate.parse_duration(f"PT{duration_seconds}S")) if duration_seconds else ""

        audio_filename = f"audio{idx:03d}.wav"
        srt_filename = f"srt{idx:03d}.srt"

        # ================= DOWNLOAD AUDIO =================
        download_cmd = [
            "yt-dlp",
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "wav",
            "-o", os.path.join(audio_folder, f"audio{idx:03d}.%(ext)s"),
            video_url
        ]

        print(f"[{idx}] 🎵 Downloading audio: {title}")
        subprocess.run(download_cmd)

        # ================= DOWNLOAD SUBTITLES =================
        try:
            transcript = download_transcript(video_id)
            formatter = SRTFormatter()
            srt_content = formatter.format_transcript(transcript)

            with open(os.path.join(srt_folder, srt_filename), "w", encoding="utf-8") as f:
                f.write(srt_content)

            print(f"[{idx}] 📝 Subtitle downloaded: {srt_filename}")
            error = ""

        except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable) as e:
            print(f"[{idx}] ⚠️ Subtitle issue: {str(e)}")
            error = str(e)

        except Exception as e:
            print(f"[{idx}] ❌ Error during subtitle download: {str(e)}")
            error = str(e)

        # ================= COLLECT METADATA =================
        videos.append({
            "AudioFile": audio_filename,
            "SubtitleFile": srt_filename if not error else "",
            "Title": title,
            "UploadDate": upload_date,
            "Duration": duration,
            "URL": video_url,
            "Error": error
        })

    except Exception as e:
        print(f"[{idx}] ❌ Failed to process video: {str(e)}")
        videos.append({
            "AudioFile": "",
            "SubtitleFile": "",
            "Title": f"Video {idx}",
            "UploadDate": "",
            "Duration": "",
            "URL": "",
            "Error": str(e)
        })

# ================= SAVE METADATA TO EXCEL =================
df = pd.DataFrame(videos)
df.to_excel("video_metadata.xlsx", index=False)
print("\n✅ All done! Metadata saved to video_metadata.xlsx")


### 7. Time-Filtered Processing (Videos Newer Than 2 Years)
**Description:** The final, production-ready pipeline for this project. It mirrors the structural stability of the previous script but introduces a strict **temporal constraint filter**:
* Uses Python's `datetime` module to dynamically compute a cutoff date corresponding to exactly 2 years ago (730 days).
* During iteration, any video uploaded prior to this cutoff date is automatically skipped (`Skipped (too old)`), conserving bandwidth and local disk storage.
* Sequentially processes eligible new videos and outputs the final curated dataset to `updated_video_metadata.xlsx`.

In [ ]:
import subprocess
import json
import isodate
import pandas as pd
import os
import re
import time
from datetime import datetime, timedelta
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import SRTFormatter
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

# =============== CONFIGURATION ===============
channel_url = input("Enter YouTube channel URL: ").strip()
audio_folder = "audios"
srt_folder = "srt"
os.makedirs(audio_folder, exist_ok=True)
os.makedirs(srt_folder, exist_ok=True)

# =============== TIME WINDOW (2 years) ===============
now = datetime.utcnow()
cutoff_date = now - timedelta(days=730)

# =============== FETCH PLAYLIST METADATA ===============
print("Fetching video metadata from channel...\n")

command = [
    "yt-dlp",
    "--flat-playlist",
    "--dump-json",
    channel_url
]

result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
video_lines = result.stdout.splitlines()

videos = []

# =============== UTILITIES ===============
def get_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([^&?/]+)", url)
    return match.group(1) if match else None

def download_transcript(video_id, language_code='ar', retries=10, delay=2):
    for attempt in range(retries):
        try:
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            for t in transcript_list:
                if t.language_code == language_code:
                    return t.fetch()
            for t in transcript_list:
                if t.language_code.startswith('a.') and 'ar' in t.language_code:
                    return t.fetch()
            raise NoTranscriptFound(video_id, language_codes=[language_code])
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                raise e

# =============== PROCESS EACH VIDEO ===============
for idx, line in enumerate(video_lines, 1):
    try:
        video_info = json.loads(line)
        video_id = video_info.get("id")
        video_url = f"https://www.youtube.com/watch?v={video_id}"

        # Detailed metadata
        detailed_cmd = [
            "yt-dlp",
            "--skip-download",
            "--dump-json",
            video_url
        ]
        detailed_result = subprocess.run(detailed_cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        detailed_info = json.loads(detailed_result.stdout)

        title = detailed_info.get("title", "Unknown Title")
        upload_date_raw = detailed_info.get("upload_date", "")
        duration_seconds = detailed_info.get("duration", 0)
        duration = str(isodate.parse_duration(f"PT{duration_seconds}S")) if duration_seconds else ""

        # Format date and filter by age
        if upload_date_raw:
            upload_date = datetime.strptime(upload_date_raw, "%Y%m%d")
            if upload_date < cutoff_date:
                print(f"[{idx}] Skipped (too old): {title}")
                continue
            formatted_date = upload_date.strftime("%Y-%m-%d")
        else:
            formatted_date = ""

        # Download audio
        audio_filename = f"audio{len(videos)+1:03d}.wav"
        srt_filename = f"srt{len(videos)+1:03d}.srt"

        audio_path = os.path.join(audio_folder, audio_filename)
        download_cmd = [
            "yt-dlp",
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "wav",
            "-o", audio_path,
            video_url
        ]
        print(f"[{len(videos)+1}] Downloading audio: {title}")
        subprocess.run(download_cmd)

        # Download SRT using API
        try:
            transcript_data = download_transcript(video_id)
            formatter = SRTFormatter()
            srt_content = formatter.format_transcript(transcript_data)
            with open(os.path.join(srt_folder, srt_filename), "w", encoding="utf-8") as f:
                f.write(srt_content)
            error_msg = ""
            print(f"[{len(videos)+1}] ✓ Subtitle saved: {srt_filename}")
        except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable) as e:
            error_msg = str(e)
            srt_filename = ""
            print(f"[{len(videos)+1}] ✗ Subtitle Error: {error_msg}")

        # Append metadata
        videos.append({
            "Title": title,
            "URL": video_url,
            "UploadDate": formatted_date,
            "Duration": duration,
            "AudioFile": audio_filename,
            "SubtitleFile": srt_filename,
            "Error": error_msg
        })

    except Exception as e:
        print(f"[{idx}] Error: {e}")
        continue

# =============== SAVE TO EXCEL ===============
df = pd.DataFrame(videos)
df.to_excel("updated_video_metadata.xlsx", index=False)
print("\n✅ All done. Metadata saved to 'updated_video_metadata.xlsx'")
